# 01 — Data Exploration

Sanity-check the trained NextTrack artifacts: catalogue size, matrix sparsity,
tag coverage, and the noisiness of Last.fm tags (the known `why`-quality
limitation). Run `uv run train` first so `prototypes/artifacts/` exists.

In [1]:
import pickle
import numpy as np
from nextrack.train import FACTORS_PATH, IDMAP_PATH

factors = np.load(FACTORS_PATH)
maps = pickle.load(open(IDMAP_PATH, 'rb'))
names = maps['track_names']
tags = maps['tags']
print('track_factors shape:', factors.shape)
print('catalogue tracks   :', f"{len(maps['track_id_to_index']):,}")
print('tracks with names  :', f'{len(names):,}')
print('tracks with tags   :', f'{len(tags):,}')

track_factors shape: (35861, 64)
catalogue tracks   : 35,861
tracks with names  : 35,861
tracks with tags   : 24,668


## Tag coverage

How many recommendable tracks can actually carry a `why` explanation?

In [2]:
n_tracks = len(maps['track_id_to_index'])
n_tagged = sum(1 for tid in maps['track_id_to_index'] if tags.get(tid))
pct = 100 * n_tagged / n_tracks
print(f'tag coverage: {n_tagged:,} / {n_tracks:,}  ({pct:.1f}%)')
print(f'-> ~{100-pct:.0f}% of recommendations will have an empty why string (honest limitation).')

tag coverage: 24,668 / 35,861  (68.8%)
-> ~31% of recommendations will have an empty why string (honest limitation).


## Top artists in the catalogue

Which artists dominate the trained slice (drives which seed sets demo well).

In [3]:
from collections import Counter
artist_counts = Counter(names[tid][0] for tid in maps['track_id_to_index'] if tid in names)
for artist, n in artist_counts.most_common(10):
    print(f'  {n:4}  {artist}')

   193  The Beatles
   114  Pink Floyd
   102  Disturbed
    91  Roxette
    90  Marilyn Manson
    89  Radiohead
    85  Lana Del Rey
    84  Metallica
    83  Muse
    80  David Bowie


## Tag noise — the `why`-quality limitation

Last.fm tags are free-text user contributions, not a clean genre vocabulary.
Some are genres (`classic rock`), some are band names (`Type O Negative`), some
are junk. We do NOT filter them — raw tags are honest about the data. A
production system would map to a controlled vocabulary (future work).

In [4]:
# Curated examples where Last.fm tags mix genres with non-genre free text:
# band names, personal annotations, emotional exclamations, bare numbers.
noisy_examples = [
    '39957893',  # Wolfheart - The Hunt
    '6152536',   # Jessie J - Bang Bang
    '41114804',  # The Cribs - Things Aren't Gonna Change
    '35730415',  # Lady Gaga - Sinner's Prayer
    '29948406',  # Heilung - Othan
]
for tid in noisy_examples:
    a, t = names[tid]
    print(f'{a} - {t}')
    print(f'    tags: {tags[tid][:6]}')

Wolfheart - The Hunt
    tags: ['Melodic Death Metal', 'cool', '2013', 'i love this fucking song', 'FUCKING AWESOME', 'i love this']
Jessie J - Bang Bang
    tags: ['pop', 'dance', 'nicki minaj', 'Ariana Grande', '2014', 'tag needs correction']
The Cribs - Things Aren't Gonna Change
    tags: ['indie', 'british i like', 'rock', 'alternative', 'post-punk', 'seen live']
Lady Gaga - Sinner's Prayer
    tags: ['country', 'pop', 'Lady Gaga', '2016', 'good as gold', 'best of 2016']
Heilung - Othan
    tags: ['folk', '1', 'neofolk', 'dark folk', 'nordic folk', '2019']
